In [1]:
import time
from datetime import datetime
import pandas as pd
import re

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import selenium.common.exceptions as se

TR_MONTHS = {
    "Ocak": 1, "Şubat": 2, "Mart": 3, "Nisan": 4, "Mayıs": 5, "Haziran": 6,
    "Temmuz": 7, "Ağustos": 8, "Eylül": 9, "Ekim": 10, "Kasım": 11, "Aralık": 12
}

date_scraped = datetime.now().strftime("%d-%m-%Y")

trendyol_base = "https://www.trendyol.com/"

def make_driver(detach=True, headless=False):
    chrome_options = webdriver.ChromeOptions()
    if detach:
        chrome_options.add_experimental_option("detach", True)

    # If you want headless:
    if headless:
        chrome_options.add_argument("--headless=new")

    # Stability options
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1400,900")

    # User-Agent (optional)
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(60)
    return driver

In [2]:
products = ["https://www.trendyol.com/fantom/pratic-s-p1200-toz-torbasiz-dikey-supurge-antrasit-p-35509789?boutiqueId=61&merchantId=110463"]

In [3]:
driver = make_driver()
wait = WebDriverWait(driver, 15)

In [4]:
driver.get(products[0])
btn = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, ".onboarding__default-renderer-primary-button"))
)
btn.click()

In [5]:
review_tag = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "a.reviews-summary-reviews-detail"))
)

href = review_tag.get_attribute("href")
driver.get(href)

In [6]:
btn = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, ".sort-dropdown-button"))
)
btn.click()

In [7]:
opt = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'div.sort-dropdown-item[data-testid="sort-option-newest"]'))
)
opt.click()

In [8]:
def parse_tr_date(day_str, month_str, year_str):
    day = int(re.sub(r"\D", "", day_str))
    month = TR_MONTHS.get(month_str.strip(), None)
    year = int(re.sub(r"\D", "", year_str))
    if not month:
        return None
    # ISO format: YYYY-MM-DD
    return datetime(year, month, day).date().isoformat()

def safe_text(el):
    try:
        return el.text.strip()
    except Exception:
        return ""

def try_find_text(parent, selectors):
    """selectors: list of CSS selectors to try in order."""
    for sel in selectors:
        try:
            found = parent.find_element(By.CSS_SELECTOR, sel)
            txt = safe_text(found)
            if txt:
                return txt
        except Exception:
            pass
    return None

def get_review_date(review_el):
    try:
        date_el = review_el.find_element(By.CSS_SELECTOR, ".review-info-detail .detail-item.date")
        spans = date_el.find_elements(By.CSS_SELECTOR, "span")
        if len(spans) >= 3:
            return parse_tr_date(spans[0].text, spans[1].text, spans[2].text)
        txt = safe_text(date_el)
        # örn: "17 Şubat 2026"
        m = re.search(r"(\d{1,2})\s+([A-Za-zÇĞİÖŞÜçğıöşü]+)\s+(\d{4})", txt)
        if m:
            return parse_tr_date(m.group(1), m.group(2), m.group(3))
    except Exception:
        pass
    return None

def get_rating(review_el):
    try:
        star_container = review_el.find_element(By.CSS_SELECTOR, '[data-testid="star-rating"]')
    except Exception:
        return None

    # 1) attribute'lar
    for attr in ["aria-label", "title", "data-rating", "data-score"]:
        try:
            val = star_container.get_attribute(attr)
            if val:
                # "4.5" / "4,5" / "4 out of 5" / "4,5 / 5" gibi şeyleri yakala
                m = re.search(r"(\d+(?:[.,]\d+)?)", val)
                if m:
                    num = float(m.group(1).replace(",", "."))
                    if 0 <= num <= 5:
                        return num
        except Exception:
            pass

    # 2) genişlik hesabı (yaklaşık)
    try:
        full = star_container.find_element(By.CSS_SELECTOR, ".star-rating-full-star")
        container_w = float(star_container.size["width"])  # px
        full_w = float(full.size["width"])

        if container_w > 0 and full_w >= 0:
            rating = 5.0 * (full_w / container_w)
            rating = round(rating * 2) / 2
            rating = max(0.0, min(5.0, rating))
            return rating
    except Exception:
        pass

    try:
        full = star_container.find_element(By.CSS_SELECTOR, ".star-rating-full-star")
        style = full.get_attribute("style") or ""
        m = re.search(r"padding-inline-end:\s*([\d.]+)px", style)
        if m:
            pad = float(m.group(1))
            container_w = float(star_container.size["width"])
            filled = max(0.0, container_w - pad)
            rating = 5.0 * (filled / container_w) if container_w > 0 else None
            if rating is not None:
                rating = round(rating * 2) / 2
                rating = max(0.0, min(5.0, rating))
                return rating
    except Exception:
        pass

    return None

def extract_one_review(review_el):
    body = try_find_text(review_el, [
        "span.review-comment",
        ".review-comment span",
        ".review-comment",
        ".comment-body span",
        ".comment-body"
    ])

    title = try_find_text(review_el, [
        "[data-testid='review-title']",
        ".review-title",
        "h3.review-title",
        "h4.review-title"
    ])

    rating = get_rating(review_el)
    review_date = get_review_date(review_el)

    return {
        "title": title,
        "rating": rating,
        "body": body,
        "review_date": review_date
    }

def load_all_reviews(driver, wait_sec=15, max_scrolls=60, sleep_between=0.8, target_reviews=80):
    wait = WebDriverWait(driver, wait_sec)

    # İlk review'ların gelmesini bekle
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.review")))

    last_count = 0
    stable_rounds = 0

    for _ in range(max_scrolls):
        cards = driver.find_elements(By.CSS_SELECTOR, "div.review")
        count = len(cards)

        if count >= target_reviews:
            break
            
        for btn_sel in [
            "button[data-testid='load-more']",
            "button.load-more",
            "button.more",
        ]:
            try:
                btn = driver.find_element(By.CSS_SELECTOR, btn_sel)
                if btn.is_displayed() and btn.is_enabled():
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(sleep_between)
            except Exception:
                pass

        # Scroll en alta
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(sleep_between)

        # tekrar say
        cards = driver.find_elements(By.CSS_SELECTOR, "div.review")
        count = len(cards)

        # yine 80'e ulaştıysak dur
        if count >= target_reviews:
            break

        # yeni review gelmiyorsa bitir
        if count == last_count:
            stable_rounds += 1
        else:
            stable_rounds = 0
            last_count = count

        if stable_rounds >= 3:
            break

def scrape_reviews_to_df(driver):
    load_all_reviews(driver)

    cards = driver.find_elements(By.CSS_SELECTOR, "div.review")
    rows = []
    for i, card in enumerate(cards):
        try:
            rows.append(extract_one_review(card))
        except StaleElementReferenceException:
            # DOM yenilendiyse tekrar bulup dene
            cards2 = driver.find_elements(By.CSS_SELECTOR, "div.review")
            if i < len(cards2):
                rows.append(extract_one_review(cards2[i]))

    df = pd.DataFrame(rows).drop_duplicates()
    return df

In [9]:
df = scrape_reviews_to_df(driver)
df.head()

  title  rating                                               body review_date
0  None     5.0   Sık sık bağlantı yerlerinin kırılmasından bıktım  2026-02-17
1  None     5.0  Çok çok beğendik. Çekim gücü çok güzel. Bizde ...  2026-02-17
2  None     5.0                                      güzel çekiyor  2026-02-17
3  None     5.0                        ürün çok güzel çok beğendim  2026-02-17
4  None     5.0                             Çök guzel...kollanışlı  2026-02-17
